<a href="https://colab.research.google.com/github/hiten4/Day38/blob/main/Day38.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Week 07 - Tuesday NLP Assignment (Word2Vec & Semantic Similarity)

Structured implementation with clear sections.

In [1]:
# Install dependencies (run once)
!pip install gensim sentence-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 47.1 MB/s eta 0:00:00


In [2]:
import pandas as pd
import numpy as np
import re
from gensim.models import Word2Vec
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer

In [3]:
# Load dataset
try:
    reviews = pd.read_csv('shopsense_reviews.csv')
    print("Data loaded")
except Exception as e:
    print("Error:", e)

reviews['review_text'] = reviews['review_text'].fillna("").astype(str)

Data loaded


In [4]:
def preprocess(text):
    text = text.lower()
    text = re.sub(r'[^a-z\s]', '', text)
    return text.split()

reviews['tokens'] = reviews['review_text'].apply(preprocess)

In [6]:
def safe_similarity(model, w1, w2):
    if w1 in model.wv and w2 in model.wv:
        return model.wv.similarity(w1, w2)
    else:
        return f"One or both words not in vocabulary: {w1}, {w2}"

print("cheap vs affordable:", safe_similarity(model, 'cheap', 'affordable'))
print("cheap vs flimsy:", safe_similarity(model, 'cheap', 'flimsy'))

cheap vs affordable: One or both words not in vocabulary: cheap, affordable
cheap vs flimsy: One or both words not in vocabulary: cheap, flimsy


In [7]:
def sentence_vector(sentence, model):
    words = preprocess(sentence)
    vecs = [model.wv[w] for w in words if w in model.wv]
    return np.mean(vecs, axis=0) if vecs else np.zeros(model.vector_size)

affordable_anchor = sentence_vector("cheap affordable price good deal", model)
quality_anchor = sentence_vector("cheap flimsy poor quality bad", model)

def classify(sentence):
    vec = sentence_vector(sentence, model)
    sim_aff = cosine_similarity([vec], [affordable_anchor])[0][0]
    sim_qual = cosine_similarity([vec], [quality_anchor])[0][0]
    return "affordable" if sim_aff > sim_qual else "low_quality"

print(classify("This product is cheap and a great deal"))
print(classify("This feels cheap and breaks easily"))

low_quality
low_quality


In [8]:
model_w2 = Word2Vec(sentences=reviews['tokens'], vector_size=100, window=2, min_count=2)
model_w10 = Word2Vec(sentences=reviews['tokens'], vector_size=100, window=10, min_count=2)

print("Window=2 (syntactic) vs Window=10 (semantic) comparison done")

Window=2 (syntactic) vs Window=10 (semantic) comparison done


In [9]:
review_a = "incredible camera but terrible battery life"
review_b = "Battery drains fast, although photos are stunning"

# BOW
def bow_vec(text, vocab):
    vec = np.zeros(len(vocab))
    for w in preprocess(text):
        if w in vocab:
            vec[vocab.index(w)] += 1
    return vec

vocab = list(set(preprocess(review_a + " " + review_b)))
bow_a = bow_vec(review_a, vocab)
bow_b = bow_vec(review_b, vocab)

print("BOW similarity:", cosine_similarity([bow_a], [bow_b])[0][0])

# TF-IDF
from sklearn.feature_extraction.text import TfidfVectorizer
vec = TfidfVectorizer()
tfidf = vec.fit_transform([review_a, review_b]).toarray()
print("TF-IDF similarity:", cosine_similarity([tfidf[0]], [tfidf[1]])[0][0])

# Word2Vec Avg
vec_a = sentence_vector(review_a, model)
vec_b = sentence_vector(review_b, model)
print("Word2Vec similarity:", cosine_similarity([vec_a], [vec_b])[0][0])

# Sentence-BERT
sbert = SentenceTransformer('all-MiniLM-L6-v2')
emb = sbert.encode([review_a, review_b])
print("SBERT similarity:", cosine_similarity([emb[0]], [emb[1]])[0][0])

BOW similarity: 0.1543033499620919
TF-IDF similarity: 0.0845798608014702
Word2Vec similarity: 0.33720642


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

SBERT similarity: 0.6335721
